In [1]:
# compute old FR average and new FR average network and nFR value 
# param nedge
import os
import sys  
import pandas as pd
import hgt_gcn
import util
import copy


hgt_file = '../../hgt_abd_metadata/ERP010700.HGT.v2.short.csv'
metadata_file = '../../hgt_abd_metadata/ERP010700.metadata.v2.short10.tsv'
abd_file = '../../hgt_abd_metadata/ERP010700.merged.short10.tsv'
sp_file = '../../hgt_abd_metadata/genome_species.tsv'
db_dir = '../../HGT_demo_file/DB.genome_annotation'
gcn_file = '../../GCN_s.tsv'
d_file = '../../sp_d.tsv'
top_n = 100
groupid = 'phenotype'
method = 'pearson'

hgt_df = pd.read_csv(hgt_file, index_col=None, header=0)
group = pd.read_csv(metadata_file, index_col=0, header=0, sep='\t')
abd_df = pd.read_csv(abd_file, index_col=0, header=0, sep='\t')
sp_df = pd.read_csv(sp_file, index_col=0, header=0, sep='\t')
gcn_df = pd.read_csv(gcn_file, index_col=0, header=0, sep='\t')
sp_d = pd.read_csv(d_file, index_col=0, header=0, sep='\t')

group = group[[groupid]]
if not util.check_valid(group, abd_df):
    exit(2)

pheno_set = list(set(group[groupid]))
pheno_samples = {}
for g in pheno_set:
    pheno_samples[g] = list(group[group[groupid] == g].index)

abd_df = hgt_gcn.multi_sample_normalize(abd_df)
genome_ko = hgt_gcn.ko_df(hgt_df, db_dir)
sp_ko_df = hgt_gcn.hgt2sp_ko(sp_df, genome_ko)
hgt_nets = hgt_gcn.hgt2sp_hgt(sp_df, genome_ko)

In [ ]:
hgt_nets

In [38]:

nfr_result_df = pd.DataFrame(columns=['sample', 'nFR', 'adj_nFR', 'group'])
sum_fr_dict = {}
sum_adj_fr_dict = {}
sum_hgt_net_dict ={}
sym_hgt_net_dict = {}

for i, g in enumerate(pheno_set):
    slist = pheno_samples[g]
    # multi sample test
    sum_fr_net = pd.DataFrame()
    sum_adj_fr_net = pd.DataFrame()
    sum_hgt_net_df = pd.DataFrame()
    for sname in slist:
        if sname in hgt_nets.keys():
            sum_hgt_net_df = hgt_gcn.net_sum(sum_hgt_net_df, hgt_nets[sname])
        nfr_result_df.loc[sname, 'sample'] = sname
        part_abd_df = abd_df[sname]
        part_abd_df = part_abd_df[part_abd_df > 0]
        tmp_abd = list(part_abd_df.index)
        part_df = sp_ko_df[sp_ko_df['sample'] == sname][['sp1', 'sp2', 'ko', 'num']]
        common_sp = list(set(tmp_abd).intersection(set(gcn_df.index)))
        tmp_d = sp_d.loc[common_sp, common_sp]
        # original fr
        nfr_value, fr_df, profile = hgt_gcn.nfr(tmp_d, abd_df, sname)
        nfr_result_df.loc[sname, 'nFR'] = nfr_value
        nfr_result_df.loc[sname, 'group'] = g
        # align and add to sum nfr net
        sum_fr_net = hgt_gcn.net_sum(sum_fr_net, fr_df)
        if len(part_df)>0:
            new_gcn_df, effect_list = hgt_gcn.hgt_adjust_gcn(gcn_df, part_df)
            effect_list = list(set(effect_list).intersection(set(common_sp)))
            tmp_gcn = gcn_df.T[common_sp]
            if len(effect_list) < 20:
                new_d = hgt_gcn.adjust_d(tmp_d, tmp_gcn, effect_list)
            else:
                new_d = hgt_gcn.make_d(new_gcn_df.loc[common_sp,])
        
            nfr_value, fr_df, profile = hgt_gcn.nfr(new_d, abd_df, sname)
            nfr_result_df.loc[sname, 'aFR'] = nfr_value
            sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
        else:
            nfr_result_df.loc[sname, 'aFR'] = nfr_value
            sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
    sum_fr_dict[g] = copy.deepcopy(sum_fr_net)
    sum_adj_fr_dict[g] = copy.deepcopy(sum_adj_fr_net)
    sum_hgt_net_dict[g] = copy.deepcopy(sum_hgt_net_df)
    common_sp = list(set(sum_fr_net.index).intersection(set(sum_hgt_net_df.index).union(set(sum_hgt_net_df.columns))))
    hgt_common_sp_index = list(set(sum_hgt_net_df.index).intersection(common_sp))
    hgt_common_sp_columns = list(set(sum_hgt_net_df.columns).intersection(common_sp))
    if len(common_sp) == 0:
        print("No HGT found for current speceis")
    sum_hgt_net_df = sum_hgt_net_df.loc[hgt_common_sp_index, hgt_common_sp_columns]
    mask, tmp = copy.deepcopy(sum_hgt_net_df).align(pd.DataFrame(0, index=common_sp, columns=common_sp), fill_value=0)
    mask = mask.loc[common_sp, common_sp]
    sym_hgt_net_dict[g] = copy.deepcopy(mask+mask.T)
    mask[mask > 0] = 1
    sum_fr_net = sum_fr_net.loc[common_sp, common_sp].multiply(mask)
    sum_adj_fr_net = sum_adj_fr_net.loc[common_sp, common_sp].multiply(mask)
    
    output_fr_net = hgt_gcn.output_fr_net(sum_fr_net, top_n)[0]
    output_adj_fr_net = hgt_gcn.output_fr_net(sum_adj_fr_net, top_n)[0]
    output_hgt_net = hgt_gcn.output_hgt_net(sum_hgt_net_df, top_n)[0]
    output_fr_net.columns = ['species1', 'species2', 'weight']
    output_adj_fr_net.columns = ['species1', 'species2', 'weight']
    output_hgt_net.columns = ['species1', 'species2', 'weight']



In [39]:
output_fr_net

,species1,species2,weight
3,s__Alistipes_senegalensis,s__Alistipes_shahii,2.399911e-05
23,s__Phocaeicola_coprocola,s__Phocaeicola_dorei,2.160329e-05
17,s__Holdemanella_sp900547815,s__Faecalibacillus_intestinalis,2.052995e-07


In [42]:
result_df = pd.DataFrame(columns=['group', 'nFR-HGT', 'aFR-HGT'])
for i, g in enumerate(pheno_set):
    result_df.loc[g, 'group'] = "{}(group{})".format(g, i+1)
    symm_hgt = sym_hgt_net_dict[g]
    result_df.loc[g, 'nFR-HGT'] = hgt_gcn.net_correlation(sum_fr_dict[g], symm_hgt, method)
    result_df.loc[g, 'aFR-HGT'] = hgt_gcn.net_correlation(sum_adj_fr_dict[g], symm_hgt, method)


In [43]:
result_df

,group,nFR-HGT,aFR-HGT
D006262,D006262(group1),-0.005508,-0.005508
D001249,D001249(group2),-0.043,-0.042516
